# I-N10 — ISLES 2022 nnU-Net + Attention Gate

이 Notebook은 기존 **I-N00 nnU-Net baseline Notebook을 기반으로** 만들었고,  
**Encoder → Decoder skip connection에 Attention Gate만 추가**한 구조 비교 실험입니다.

- Dataset: `Dataset502_ISLES2022`
- Input: DWI + ADC (2 channels)
- Split: Train 175 / Validation 38 / Test 37
- Configuration: `3d_fullres`, Fold 0
- Patch: `80 × 96 × 80`
- Batch size: `6`
- Epoch: 최대 300
- Early Stopping: patience 30 / min_delta 0.001
- **변경점: Attention Gate만 추가**
- Checkpoint: `brainCDSS/checkpoints/I-N10/`

> 기존 I-N00 baseline checkpoint와 결과는 수정하거나 덮어쓰지 않습니다.


In [2]:
!nvidia-smi

Thu Sep  3 12:42:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# =========================================================
# GitHub model 저장소(dev) 준비
#
# - fresh Colab runtime에서는 저장소를 clone
# - 이미 /content/model에 저장소가 있으면 다시 clone하지 않음
# - GitHub Token은 Colab Secret의 "git_token"에서 읽음
# - Token 문자열은 Notebook에 직접 저장하지 않음
# =========================================================

from google.colab import userdata
import os
import subprocess
import tempfile

repo_dir = "/content/model"

# ---------------------------------------------------------
# 1. 이미 clone된 저장소인지 확인
# ---------------------------------------------------------
if os.path.exists(os.path.join(repo_dir, ".git")):
    print("model repository already exists:", repo_dir)

# ---------------------------------------------------------
# 2. 저장소가 없으면 dev 브랜치 clone
# ---------------------------------------------------------
else:
    # Colab Secret에서 GitHub Token 읽기
    token = userdata.get("git_token")

    if not token:
        raise RuntimeError(
            "Colab Secret에 'git_token'이 등록되어 있지 않습니다."
        )

    # Git 인증용 임시 AskPass 파일 생성
    with tempfile.NamedTemporaryFile(
        mode="w",
        delete=False,
        suffix=".sh",
    ) as f:
        f.write(
            '#!/bin/sh\n'
            'case "$1" in\n'
            '  *Username*) echo "x-access-token" ;;\n'
            '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
            'esac\n'
        )
        askpass = f.name

    os.chmod(askpass, 0o700)

    env = os.environ.copy()
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = askpass
    env["GIT_TERMINAL_PROMPT"] = "0"

    try:
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                "dev",
                "https://github.com/brainCDSS/model.git",
                repo_dir,
            ],
            env=env,
            check=True,
        )

    finally:
        # 인증용 임시 파일 제거
        if os.path.exists(askpass):
            os.remove(askpass)

        del token

# ---------------------------------------------------------
# 3. 현재 저장소 상태 확인
# ---------------------------------------------------------
os.chdir(repo_dir)

print("\n현재 위치:", os.getcwd())

print("\n현재 브랜치:")
subprocess.run(
    ["git", "branch", "--show-current"],
    check=True,
)

print("\n현재 최신 커밋:")
subprocess.run(
    ["git", "log", "-1", "--oneline"],
    check=True,
)


현재 위치: /content/model

현재 브랜치:

현재 최신 커밋:


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [4]:
# =========================================================
# Google Cloud 인증
#
# ISLES 2022 전처리 데이터는 GCS에 저장되어 있으므로
# Colab 런타임 시작 시 Google Cloud 계정 인증 필요
#
# ※ GitHub Token 인증과 별개임
# =========================================================

from google.colab import auth

auth.authenticate_user()

print("Google Cloud 인증 완료")



Google Cloud 인증 완료


In [5]:
# =========================================================
# ISLES 2022 nnU-Net 전처리 데이터 위치 확인
#
# 실제 학습 데이터는 GitHub가 아니라 GCS에서 관리
# I-N10은 아래 nnU-Net preprocessing 결과를 사용
# =========================================================

!gcloud storage ls gs://brainon-cdss-datasets/ischemia/v1/nnunet/

gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/
gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_raw/


In [6]:
# =========================================================
# nnU-Net Colab 작업 경로 설정
#
# - GCS의 전처리 데이터를 Colab 로컬 캐시에 복사해 학습
# - /content 영역은 런타임 종료 시 삭제될 수 있음
# - 학습 checkpoint는 Google Drive에 별도 백업
# =========================================================

import os

# Colab 학습용 로컬 캐시
CACHE_ROOT = "/content/brainon_cache"

os.environ["BRAINON_CACHE_ROOT"] = CACHE_ROOT
os.environ["nnUNet_raw"] = f"{CACHE_ROOT}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{CACHE_ROOT}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{CACHE_ROOT}/nnUNet_results"

# 필요한 폴더 생성
for path in [
    os.environ["nnUNet_raw"],
    os.environ["nnUNet_preprocessed"],
    os.environ["nnUNet_results"],
]:
    os.makedirs(path, exist_ok=True)

print("nnUNet_raw =", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results =", os.environ["nnUNet_results"])

nnUNet_raw = /content/brainon_cache/nnUNet_raw
nnUNet_preprocessed = /content/brainon_cache/nnUNet_preprocessed
nnUNet_results = /content/brainon_cache/nnUNet_results


In [7]:
!gcloud storage cp --recursive \
  gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022 \
  /content/brainon_cache/nnUNet_preprocessed/

Copying gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/dataset.json to file:///content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/dataset.json
Copying gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/dataset_fingerprint.json to file:///content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/dataset_fingerprint.json
Copying gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations/sub-strokecase0001.nii.gz to file:///content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations/sub-strokecase0001.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations/sub-strokecase0002.nii.gz to file:///content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations/sub-strokecase0002.nii.gz
Copying gs://brainon-cdss-datasets/ischemia/v1/nnunet/nnUNet_preprocessed/Da

In [8]:
# =========================================================
# ISLES 2022 nnU-Net 전처리 데이터 검증
#
# GCS에서 Colab 로컬 캐시로 복사한 Dataset502_ISLES2022가
# 학습에 필요한 파일 구조를 정상적으로 갖추고 있는지 확인
#
# 확인 항목
# - dataset.json
# - dataset_fingerprint.json
# - nnUNetPlans.json
# - splits_final.json
# - 3d_fullres 전처리 데이터(.b2nd)
# - Case별 properties 파일(.pkl)
# - Fold 0 Train / Validation Case 수
#
# ※ 실제 학습은 아래 Colab 캐시의 전처리 데이터를 사용
# =========================================================

from pathlib import Path
import json

root = Path(
    "/content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022"
)

print("Dataset 폴더:", root.exists())
print("dataset.json:", (root / "dataset.json").exists())
print("dataset_fingerprint.json:", (root / "dataset_fingerprint.json").exists())
print("nnUNetPlans.json:", (root / "nnUNetPlans.json").exists())
print("splits_final.json:", (root / "splits_final.json").exists())

fullres = root / "nnUNetPlans_3d_fullres"

print("\n3d_fullres 폴더:", fullres.exists())
print("data(.b2nd):", len(list(fullres.glob("sub-strokecase*.b2nd"))))
print("properties(.pkl):", len(list(fullres.glob("*.pkl"))))

with open(root / "splits_final.json", encoding="utf-8") as f:
    splits = json.load(f)

print("\nFold 수:", len(splits))
print("Fold 0 Train:", len(splits[0]["train"]))
print("Fold 0 Val:", len(splits[0]["val"]))

Dataset 폴더: True
dataset.json: True
dataset_fingerprint.json: True
nnUNetPlans.json: True
splits_final.json: True

3d_fullres 폴더: True
data(.b2nd): 426
properties(.pkl): 213

Fold 수: 1
Fold 0 Train: 175
Fold 0 Val: 38


In [9]:
# =========================================================
# I-N10 학습 환경 사전 점검 (Preflight Check)
#
# 학습 환경 구성을 진행하기 전에 다음 항목을 확인
# - GitHub model 저장소 및 dev branch
# - Dataset502_ISLES2022 전처리 데이터
# - nnU-Net split 및 3d_fullres 데이터
# - Colab CUDA / GPU 사용 가능 여부
# =========================================================

from pathlib import Path
import subprocess
import torch

repo = Path("/content/model")
data = Path(
    "/content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022"
)

print("=== GitHub 코드 ===")
print("/content/model 존재:", repo.exists())

if (repo / ".git").exists():
    branch = subprocess.check_output(
        ["git", "-C", str(repo), "branch", "--show-current"],
        text=True
    ).strip()
    print("Git branch:", branch)
else:
    print("Git 저장소 확인 필요")

print("\n=== nnU-Net 데이터 ===")
print("Dataset502 존재:", data.exists())
print("splits_final.json:", (data / "splits_final.json").exists())
print(
    "3d_fullres b2nd:",
    len(list((data / "nnUNetPlans_3d_fullres").glob("*.b2nd")))
    if data.exists() else 0
)

print("\n=== GPU ===")
print("CUDA:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available() else "None"
)

=== GitHub 코드 ===
/content/model 존재: True
Git branch: dev

=== nnU-Net 데이터 ===
Dataset502 존재: True
splits_final.json: True
3d_fullres b2nd: 426

=== GPU ===
CUDA: True
GPU: Tesla T4


In [10]:
# I-N10 nnU-Net 학습 환경 요구사항 확인

%cd /content/model

!grep -n "requires-python" pyproject.toml
!echo "----- nnU-Net requirements -----"
!cat requirements-nnunet-cu130.txt

/content
10:requires-python = ">=3.10,<3.13"
----- nnU-Net requirements -----
# Target environment: Python 3.12 / Windows / NVIDIA RTX 4060 Ti / CUDA 13.0
# The CUDA-enabled PyTorch wheel supplies its matching CUDA runtime libraries.
--extra-index-url https://download.pytorch.org/whl/cu130

# Core frameworks
torch==2.13.0+cu130
nnunetv2==2.8.1
monai==1.6.0

# Numerical / data
numpy==2.5.2
scipy==1.18.1
pandas==3.0.5
scikit-learn==1.9.0
scikit-image==0.26.0

# Medical imaging
SimpleITK==2.5.6
nibabel==5.4.2
ImageIO==2.37.4

# nnU-Net dependencies
batchgenerators==0.25.3
batchgeneratorsv2==0.3.5
acvl_utils==0.2.6
dynamic_network_architectures==0.4.4
connected-components-3d==4.1.0
blosc2==4.11.0

# Training utilities
tqdm==4.70.0
tensorboard==2.21.0
einops==0.8.2
timm==1.0.22

# Visualization and notebook monitoring
matplotlib==3.11.1
seaborn==0.13.2
jupyterlab>=4,<5
ipywidgets>=8,<9

# Config / utility
PyYAML==6.0.3
packaging==26.3


In [11]:
# Python 3.12 학습환경 생성
!python -m pip install -q uv

# Python 3.12 설치
!uv python install 3.12

# brainCDSS 모델 학습용 가상환경 생성
!uv venv /content/brainon_py312 --python 3.12

# 버전 확인
!/content/brainon_py312/bin/python --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 92.3 MB/s eta 0:00:00
Installed Python 3.12.14 in 2.20s
 + cpython-3.12.14-linux-x86_64-gnu (python3.12)
Using CPython 3.12.14
Creating virtual environment at: /content/brainon_py312
Activate with: source /content/brainon_py312/bin/activate
Python 3.12.14


In [12]:
from pathlib import Path

print("/content/model 존재:",
      Path("/content/model").exists())

print("requirements 존재:",
      Path("/content/model/requirements-nnunet-cu130.txt").exists())

/content/model 존재: True
requirements 존재: True


In [13]:
!uv pip install \
  --python /content/brainon_py312/bin/python \
  --index-strategy unsafe-best-match \
  -r /content/model/requirements-nnunet-cu130.txt

Using Python 3.12.14 environment at: /content/brainon_py312
Resolved 185 packages in 4.74s
Prepared 185 packages in 59.57s
Installed 185 packages in 1.43s
 + absl-py==2.5.0
 + acvl-utils==0.2.6
 + annotated-types==0.8.0
 + anyio==4.15.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + argparse==1.4.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.3.0
 + attrs==26.1.0
 + babel==2.18.0
 + batchgenerators==0.25.3
 + batchgeneratorsv2==0.3.5
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + blosc2==4.11.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + comm==0.2.3
 + connected-components-3d==4.1.0
 + contourpy==1.3.3
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.8.1
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + debugpy==1.8.21
 + defusedxml==0.7.1
 + dynamic-network-architectures==0.4.4
 + einops==0.8.2
 + executing==2.2.1
 + fastjsonschema==2.22.2
 + filelock==3.32.5
 + fonttools==4.64.0
 + fqdn==1.5.1
 + fsspec==2026.

In [14]:
# =========================================================
# I-N10 nnU-Net 학습 환경 검증
#
# Python 3.12 가상환경에 설치된 주요 패키지 버전과
# CUDA/GPU 사용 가능 여부를 확인
#
# 마지막에는 실제 CUDA tensor 연산을 수행하여
# GPU 학습 환경이 정상적으로 구성됐는지 검증
# =========================================================

%%bash

/content/brainon_py312/bin/python - <<'PY'
import sys
import torch
import monai
import numpy as np
from importlib.metadata import version

print("=== Environment ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("nnUNetv2:", version("nnunetv2"))
print("MONAI:", monai.__version__)
print("NumPy:", np.__version__)

print("\n=== GPU ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

    # 실제 GPU 연산 확인
    x = torch.randn(512, 512, device="cuda")
    y = x @ x

    print("CUDA tensor:", y.device)
    print("CUDA operation: PASS")
PY

=== Environment ===
Python: 3.12.14
PyTorch: 2.13.0+cu130
PyTorch CUDA: 13.0
nnUNetv2: 2.8.1
MONAI: 1.6.0
NumPy: 2.5.2

=== GPU ===
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
VRAM: 14.56 GB
CUDA tensor: cuda:0
CUDA operation: PASS


In [15]:
# =========================================================
# brainCDSS 모델 패키지 설치
#
# GitHub에서 clone한 /content/model 소스 코드를
# Python 3.12 학습환경에 editable mode로 설치
#
# 외부 dependency는 앞 단계에서 이미 설치했으므로
# --no-deps 옵션 사용
# =========================================================

!uv pip install \
  --python /content/brainon_py312/bin/python \
  --no-deps \
  -e /content/model

Using Python 3.12.14 environment at: /content/brainon_py312
Resolved 1 package in 1ms
Prepared 1 package in 594ms
Installed 1 package in 1ms
 + brainon-ai==0.1.0 (from file:///content/model)


In [16]:
# =========================================================
# I-N10 Attention Gate repository source 확인
#
# 전용 YAML / network / Trainer는 repository 정식 파일을 사용합니다.
# 이 셀의 기존 output은 실제 학습 notebook의 provenance로 보존합니다.
# =========================================================

from pathlib import Path

repo = Path("/content/model")
attention_cfg = repo / "configs/experiments/ischemia/I-N10.yaml"
network_path = repo / "src/brainon_ai/nnunet/networks/attention_unet.py"
trainer_path = (
    repo
    / "src/brainon_ai/nnunet/trainers"
    / "nnUNetTrainerIschemiaAttention.py"
)

assert attention_cfg.is_file()
assert network_path.is_file()
assert trainer_path.is_file()

from brainon_ai.nnunet.networks.attention_unet import (
    AdditiveAttentionGate,
    AttentionUNetDecoder,
    attach_attention_decoder,
)
from brainon_ai.nnunet.trainers.nnUNetTrainerIschemiaAttention import (
    nnUNetTrainerIschemiaAttention,
)

print("I-N10 repository source: PASS")
print("Config :", attention_cfg)
print("Network:", network_path)
print("Trainer:", trainer_path)

In [15]:
# =========================================================
# I-N10 학습 실행 전 최종 환경 점검
#
# - brainon_ai 패키지 import
# - nnUNetv2 설치 상태
# - Dataset502 전처리 데이터
# - 3d_fullres / split 파일
# - nnUNetv2_train 실행 명령
#
# 실제 학습을 시작하기 전에 필요한 구성요소를 최종 확인
# =========================================================

%%bash

export nnUNet_raw=/content/brainon_cache/nnUNet_raw
export nnUNet_preprocessed=/content/brainon_cache/nnUNet_preprocessed
export nnUNet_results=/content/brainon_cache/nnUNet_results

echo "=== brainon-ai ==="
/content/brainon_py312/bin/python -c "import brainon_ai; print('brainon_ai import: PASS')"

echo
echo "=== nnU-Net ==="
/content/brainon_py312/bin/python -c "from importlib.metadata import version; print('nnunetv2:', version('nnunetv2'))"

echo
echo "=== Dataset502 ==="
/content/brainon_py312/bin/python - <<'PY'
from pathlib import Path

p = Path("/content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022")
print("Dataset502:", p.exists())
print("3d_fullres:", (p / "nnUNetPlans_3d_fullres").exists())
print("splits_final:", (p / "splits_final.json").exists())
PY

echo
echo "=== nnUNet train command ==="
test -x /content/brainon_py312/bin/nnUNetv2_train \
  && echo "nnUNetv2_train: PASS" \
  || echo "nnUNetv2_train: NOT FOUND"

=== brainon-ai ===

=== nnU-Net ===
nnunetv2: 2.8.1

=== Dataset502 ===
Dataset502: True
3d_fullres: True
splits_final: True

=== nnUNet train command ===
nnUNetv2_train: PASS


Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'brainon_ai'


In [16]:
# =========================================================
# Google Drive 마운트
#
# I-N10 학습 checkpoint를 런타임 종료 후에도
# 보존하기 위해 Google Drive 연결
# =========================================================

from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [17]:
!ls -lh "/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/fold_0"

total 382M
-rw------- 1 root root 128M Sep  3 02:38 checkpoint_best.pth
-rw------- 1 root root 128M Sep  3 07:02 checkpoint_final.pth
-rw------- 1 root root 128M Sep  3 07:03 checkpoint_latest.pth
-rw------- 1 root root  157 Sep  3 07:02 early_stopping_state.json


In [20]:
# Checkpoint 저장 경로 설정

from pathlib import Path

checkpoint_dir = Path(
    "/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10"
)

checkpoint_dir.mkdir(parents=True, exist_ok=True)

print("Checkpoint directory:", checkpoint_dir)
print("Exists:", checkpoint_dir.exists())

Checkpoint directory: /content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10
Exists: True


In [21]:
from pathlib import Path
import shutil

# I-N10 첫 실행이면 checkpoint가 아직 없습니다.
# 기존 I-N10 checkpoint가 있을 때만 local 결과 폴더로 복구합니다.
src = Path(
    "/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/fold_0"
)

dst = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0"
)

dst.mkdir(parents=True, exist_ok=True)

files = [
    "checkpoint_best.pth",
    "checkpoint_final.pth",
    "checkpoint_latest.pth",
    "early_stopping_state.json",
]

if not src.exists():
    print("I-N10 첫 실행: 기존 Drive checkpoint 없음 → 새 학습으로 시작")
else:
    restored = 0
    for name in files:
        source = src / name
        if source.exists():
            shutil.copy2(source, dst / name)
            print(name, "복구 완료")
            restored += 1
    print(f"\n복구 파일 수: {restored}")

print("Local result folder:", dst)


I-N10 첫 실행: 기존 Drive checkpoint 없음 → 새 학습으로 시작
Local result folder: /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0


In [ ]:
# =========================================================
# I-N10 Custom Trainer / Attention Gate 설정 검증
# =========================================================

%%bash

export nnUNet_raw=/content/brainon_cache/nnUNet_raw
export nnUNet_preprocessed=/content/brainon_cache/nnUNet_preprocessed
export nnUNet_results=/content/brainon_cache/nnUNet_results

export nnUNet_extTrainer=/content/model/src/brainon_ai/nnunet/trainers
export BRAINON_EXPERIMENT_CONFIG=/content/model/configs/experiments/ischemia/I-N10.yaml
export BRAINON_CHECKPOINT_DIR=/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10

# 구조 확인 과정에서만 compile 비활성화
export nnUNet_compile=false

/content/brainon_py312/bin/python - <<'PY'
import gc
import torch
from nnunetv2.run.run_training import get_trainer_from_args

trainer = get_trainer_from_args(
    "502",
    "3d_fullres",
    0,
    trainer_name="nnUNetTrainerIschemiaAttention",
    plans_identifier="nnUNetPlans",
    continue_training=False,
    device=torch.device("cuda"),
)

trainer.initialize()

print("\n=== I-N10 TRAINER CHECK ===")
print("Trainer       :", trainer.__class__.__name__)
print("Epochs        :", trainer.num_epochs)
print("Patience      :", trainer.early_stopping_patience)
print("Min delta     :", trainer.early_stopping_min_delta)
print("Patch size    :", list(trainer.configuration_manager.patch_size))
print("Batch size    :", trainer.configuration_manager.batch_size)
print("Output folder :", trainer.output_folder)
print("Drive folder  :", trainer.persistent_dir)
print("Network       :", trainer.network.__class__.__name__)
print("Decoder       :", trainer.network.decoder.__class__.__name__)
print("Attention gates:", len(trainer.network.decoder.attention_gates))

assert trainer.__class__.__name__ == "nnUNetTrainerIschemiaAttention"
assert trainer.num_epochs == 300
assert trainer.early_stopping_patience == 30
assert trainer.early_stopping_min_delta == 0.001
assert list(trainer.configuration_manager.patch_size) == [80, 96, 80]
assert trainer.configuration_manager.batch_size == 6
assert trainer.network.decoder.__class__.__name__ == "AttentionUNetDecoder"
assert len(trainer.network.decoder.attention_gates) == 4

print("\nI-N10 ATTENTION TRAINER CONFIG: PASS")

del trainer
gc.collect()
torch.cuda.empty_cache()
PY


In [23]:
%%bash

# =========================================================
# I-N10 학습 결과 및 Checkpoint 백업 확인
#
# - Colab 로컬 nnUNet_results 확인
# - Google Drive checkpoint 백업 확인
# - 최근 training log 확인
# =========================================================

RESULT="/content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0"
DRIVE="/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/fold_0"


echo "=== LOCAL RESULT ==="
if [ -d "$RESULT" ]; then
    ls -lah "$RESULT"
else
    echo "현재 런타임에 local 결과 폴더 없음"
fi

echo
echo "=== DRIVE CHECKPOINT ==="
if [ -d "$DRIVE" ]; then
    ls -lah "$DRIVE"
else
    echo "Drive checkpoint 폴더 없음"
fi

echo
echo "=== TRAINING LOG ==="
if [ -d "$RESULT" ]; then
    find "$RESULT" \
        -maxdepth 1 \
        -type f \
        -name "*.txt" \
        -print \
        -exec tail -n 20 {} \;
else
    echo "현재 런타임에 확인할 training log 없음"
fi

=== LOCAL RESULT ===
total 255M
drwxr-xr-x 2 root root 4.0K Sep  3 05:37 .
drwxr-xr-x 3 root root 4.0K Sep  3 05:36 ..
-rw------- 1 root root 128M Sep  3 02:38 checkpoint_best.pth
-rw------- 1 root root 128M Sep  3 03:07 checkpoint_latest.pth
-rw------- 1 root root  156 Sep  3 03:07 early_stopping_state.json
-rw-r--r-- 1 root root  404 Sep  3 05:37 training_log_2026_9_3_05_37_49.txt

=== DRIVE CHECKPOINT ===
total 255M
-rw------- 1 root root 128M Sep  3 02:38 checkpoint_best.pth
-rw------- 1 root root 128M Sep  3 03:07 checkpoint_latest.pth
-rw------- 1 root root  156 Sep  3 03:07 early_stopping_state.json

=== TRAINING LOG ===
/content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0/training_log_2026_9_3_05_37_49.txt

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021).

In [24]:
import os
import re
import subprocess
from pathlib import Path
from tqdm.notebook import tqdm

# =========================================================
# I-N10 학습 / 자동 Resume
# =========================================================

env = os.environ.copy()

env["nnUNet_raw"] = "/content/brainon_cache/nnUNet_raw"
env["nnUNet_preprocessed"] = "/content/brainon_cache/nnUNet_preprocessed"
env["nnUNet_results"] = "/content/brainon_cache/nnUNet_results"

env["nnUNet_extTrainer"] = (
    "/content/model/src/brainon_ai/nnunet/trainers"
)

env["BRAINON_EXPERIMENT_CONFIG"] = (
    "/content/model/configs/experiments/ischemia/I-N10.yaml"
)

env["BRAINON_CHECKPOINT_DIR"] = (
    "/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10"
)

env["PYTHONUNBUFFERED"] = "1"

drive_latest = Path(
    "/content/gdrive/MyDrive/brainCDSS/checkpoints/"
    "I-N10/fold_0/checkpoint_latest.pth"
)

cmd = [
    "/content/brainon_py312/bin/nnUNetv2_train",
    "502",
    "3d_fullres",
    "0",
    "-tr", "nnUNetTrainerIschemiaAttention",
    "-p", "nnUNetPlans",
    "-device", "cuda",
]

if drive_latest.exists():
    cmd.append("--c")
    run_mode = "Resume"
else:
    run_mode = "New training"

print("=" * 60)
print("I-N10 nnU-Net + Attention Gate")
print("Mode       :", run_mode)
print("Dataset    : Dataset502_ISLES2022")
print("Train / Val: 175 / 38")
print("Patch      : 80 x 96 x 80")
print("Batch      : 6")
print("Checkpoint :", env["BRAINON_CHECKPOINT_DIR"])
print("=" * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

progress = tqdm(
    total=250,
    desc="학습 준비 중",
    unit="it",
)

current_epoch = None
phase = None

for raw_line in iter(process.stdout.readline, ""):
    line = raw_line.strip()

    if line.startswith("[Resume]"):
        print(line)
        continue

    epoch_match = re.fullmatch(r"Epoch\s+(\d+)/300", line)
    if epoch_match:
        current_epoch = int(epoch_match.group(1))
        continue

    if line == "[TRAIN]":
        phase = "TRAIN"
        progress.reset(total=250)
        progress.set_description(f"Epoch {current_epoch}/300 | TRAIN")
        progress.set_postfix_str("시작")
        continue

    if line == "[VALIDATION]":
        phase = "VAL"
        progress.reset(total=50)
        progress.set_description(f"Epoch {current_epoch}/300 | VAL")
        progress.set_postfix_str("시작")
        continue

    if phase is not None and line.startswith("["):
        progress_match = re.search(
            r"\]\s*(\d+)/(\d+)\s*\(\s*[\d.]+%\)"
            r".*?loss\s+([-\d.]+)"
            r".*?avg\s+([-\d.]+)"
            r".*?ETA\s+(\S+)",
            line,
        )

        if progress_match:
            current = int(progress_match.group(1))
            total = int(progress_match.group(2))
            loss = progress_match.group(3)
            avg_loss = progress_match.group(4)
            eta = progress_match.group(5)

            progress.total = total
            progress.n = current
            progress.set_description(
                f"Epoch {current_epoch}/300 | {phase}"
            )
            progress.set_postfix_str(
                f"loss={loss} | avg={avg_loss} | ETA={eta}"
            )
            progress.refresh()
            continue

    important_results = (
        "완료",
        "Train Loss",
        "Val Loss",
        "Validation Dice",
        "Best Dice",
        "Early Stopping",
        "Learning Rate",
        "Epoch Time",
        "Estimated Left",
        "New Best",
        "checkpoint_latest",
        "Early Stopping triggered",
    )

    if any(key in line for key in important_results):
        print(line)

process.stdout.close()
return_code = process.wait()
progress.close()

print()
print("학습 프로세스 종료 코드:", return_code)

if return_code != 0:
    raise RuntimeError(
        "I-N10 학습 프로세스가 오류로 종료되었습니다. "
        "마지막 출력 로그를 확인하세요."
    )


I-N10 nnU-Net + Attention Gate
Mode       : Resume
Dataset    : Dataset502_ISLES2022
Train / Val: 175 / 38
Patch      : 80 x 96 x 80
Batch      : 6
Checkpoint : /content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10


학습 준비 중:   0%|          | 0/250 [00:00<?, ?it/s]

[Resume] 현재 Epoch: 133
[Resume] Best Dice: 0.8747
[Resume] Early Stopping: 8/30
Epoch 134/300 완료
Train Loss        : -0.8117
Val Loss          : -0.7604
Validation Dice   : 0.8478
Best Dice         : 0.8747 (Epoch 125)
Early Stopping    : 9/30
Learning Rate     : 0.00590250
Epoch Time        : 00:05:38
Estimated Left    : 15:36:05
✓ checkpoint_latest.pth → Google Drive 백업 완료
Epoch 135/300 완료
Train Loss        : -0.8031
Val Loss          : -0.7492
Validation Dice   : 0.8247
Best Dice         : 0.8747 (Epoch 125)
Early Stopping    : 10/30
Learning Rate     : 0.00587068
Epoch Time        : 00:03:42
Estimated Left    : 12:51:41
✓ checkpoint_latest.pth → Google Drive 백업 완료
Epoch 136/300 완료
Train Loss        : -0.8112
Val Loss          : -0.7443
Validation Dice   : 0.8038
Best Dice         : 0.8747 (Epoch 125)
Early Stopping    : 11/30
Learning Rate     : 0.00583884
Epoch Time        : 00:03:39
Estimated Left    : 11:51:19
✓ checkpoint_latest.pth → Google Drive 백업 완료
Epoch 137/300 완료
Train L

In [25]:
import json
from pathlib import Path

candidates = list(Path("/content").rglob("dataset.json"))

for p in candidates:
    if "Dataset502_ISLES2022" in str(p):
        print("PATH:", p)
        with open(p, "r") as f:
            data = json.load(f)
        print("channel_names:", data.get("channel_names"))

PATH: /content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/dataset.json
channel_names: {'0': 'DWI', '1': 'ADC'}
PATH: /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/dataset.json
channel_names: {'0': 'DWI', '1': 'ADC'}


In [26]:
%%bash

grep -R -l \
"class nnUNetTrainerIschemiaAttention" \
/content/model \
/content/src \
/content/brainon* \
2>/dev/null | head -20

/content/model/src/brainon_ai/nnunet/trainers/nnUNetTrainerIschemiaAttention.py


In [27]:
from pathlib import Path
import shutil

PRE = Path("/content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022")

MODEL = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres"
)

MODEL.mkdir(parents=True, exist_ok=True)

# dataset.json 복사
src_dataset = PRE / "dataset.json"
dst_dataset = MODEL / "dataset.json"

if not src_dataset.exists():
    raise FileNotFoundError(src_dataset)

shutil.copy2(src_dataset, dst_dataset)

# nnUNetPlans.json -> plans.json 으로 복사
src_plans = PRE / "nnUNetPlans.json"
dst_plans = MODEL / "plans.json"

if not src_plans.exists():
    raise FileNotFoundError(src_plans)

shutil.copy2(src_plans, dst_plans)

print("=== 추론용 메타데이터 준비 완료 ===")
print("dataset.json :", dst_dataset.exists(), dst_dataset)
print("plans.json   :", dst_plans.exists(), dst_plans)

ckpt = MODEL / "fold_0" / "checkpoint_best.pth"
print("checkpoint   :", ckpt.exists(), ckpt)

=== 추론용 메타데이터 준비 완료 ===
dataset.json : True /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/dataset.json
plans.json   : True /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/plans.json
checkpoint   : True /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0/checkpoint_best.pth


In [28]:
from pathlib import Path

drive_ckpt = Path(
    "/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/fold_0/checkpoint_best.pth"
)

print("Drive best checkpoint:", drive_ckpt.exists())
print("경로:", drive_ckpt)

Drive best checkpoint: True
경로: /content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/fold_0/checkpoint_best.pth


In [29]:
from pathlib import Path

fold = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0"
)

print("=== checkpoint 확인 ===")
for f in sorted(fold.glob("checkpoint*.pth")):
    print(f.name)

=== checkpoint 확인 ===
checkpoint_best.pth
checkpoint_final.pth


In [30]:
%%bash

export nnUNet_raw=/content/brainon_cache/nnUNet_raw
export nnUNet_preprocessed=/content/brainon_cache/nnUNet_preprocessed
export nnUNet_results=/content/brainon_cache/nnUNet_results

export nnUNet_extTrainer=/content/model/src/brainon_ai/nnunet/trainers

export BRAINON_EXPERIMENT_CONFIG=/content/model/configs/experiments/ischemia/I-N10.yaml

export BRAINON_CHECKPOINT_DIR=/content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10

echo "=== nnU-Net Validation 시작 ==="

/content/brainon_py312/bin/nnUNetv2_train \
  502 \
  3d_fullres \
  0 \
  -tr nnUNetTrainerIschemiaAttention \
  -p nnUNetPlans \
  --val \
  --val_best \
  -device cuda

=== nnU-Net Validation 시작 ===

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Trainer 'nnUNetTrainerIschemiaAttention' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /content/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerIschemiaAttention in folder /content/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
  Inspecting module: nnUNetTrainerAneurysmEarlyStopping
  Inspecting module: nnUNetTrainerIschemiaAttention
Found class nnUNetTrainerIschemiaAttention in nnUNetTrainerIschemiaAttention
Using trainer 'nnUNetTrainerIschemiaAttention' from: /content/model/src/brainon_ai/nnunet/trainers


In [31]:
from pathlib import Path

VAL_DIR = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0/validation"
)

print("Validation 폴더:", VAL_DIR.exists())

for f in VAL_DIR.glob("*.json"):
    print(f.name)

Validation 폴더: True
summary.json


In [32]:
import json
from pathlib import Path

VAL_DIR = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0/validation"
)

summary_path = VAL_DIR / "summary.json"

with open(summary_path, "r") as f:
    summary = json.load(f)

print("summary keys:", summary.keys())
print("case 수:", len(summary.get("metric_per_case", [])))
print()

# 첫 번째 환자 결과 구조 확인
print(json.dumps(summary["metric_per_case"][0], indent=2)[:3000])

summary keys: dict_keys(['foreground_mean', 'mean', 'metric_per_case'])
case 수: 38

{
  "metrics": {
    "1": {
      "Dice": 0.8137931034482758,
      "FN": 0,
      "FP": 54,
      "IoU": 0.686046511627907,
      "TN": 902996,
      "TP": 118,
      "n_pred": 172,
      "n_ref": 118
    }
  },
  "prediction_file": "/content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0/validation/sub-strokecase0002.nii.gz",
  "reference_file": "/content/brainon_cache/nnUNet_preprocessed/Dataset502_ISLES2022/gt_segmentations/sub-strokecase0002.nii.gz"
}


In [33]:
import json
import numpy as np
import pandas as pd
import nibabel as nib
from pathlib import Path

VAL_DIR = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0/validation"
)

summary_path = VAL_DIR / "summary.json"

with open(summary_path, "r") as f:
    summary = json.load(f)

rows = []

for item in summary["metric_per_case"]:
    m = item["metrics"]["1"]

    ref_path = Path(item["reference_file"])
    case_id = ref_path.name.replace(".nii.gz", "")

    dice = m["Dice"]
    n_ref = m["n_ref"]
    n_pred = m["n_pred"]

    # 실제 voxel 크기(mm) 확인
    nii = nib.load(ref_path)
    spacing = nii.header.get_zooms()[:3]
    voxel_volume_mm3 = np.prod(spacing)

    # 실제 병변 부피 mL
    lesion_volume_ml = n_ref * voxel_volume_mm3 / 1000.0

    # 병변 크기 그룹
    if n_ref == 0:
        size_group = "Empty-GT"
    elif lesion_volume_ml < 5:
        size_group = "Small"
    elif lesion_volume_ml < 20:
        size_group = "Medium"
    else:
        size_group = "Large"

    rows.append({
        "case": case_id,
        "dice": dice,
        "gt_voxels": n_ref,
        "pred_voxels": n_pred,
        "lesion_volume_ml": lesion_volume_ml,
        "size_group": size_group
    })

df_val = pd.DataFrame(rows)

# 병변 존재 Case
lesion_df = df_val[df_val["gt_voxels"] > 0].copy()

print("========================================")
print(" nnU-Net VALIDATION CASE-WISE RESULT")
print("========================================")
print("전체 Validation Case :", len(df_val))
print("병변 존재 Case       :", len(lesion_df))
print("Empty-GT Case        :", (df_val["gt_voxels"] == 0).sum())

print("\n환자별 Dice")
print(
    df_val[
        ["case", "lesion_volume_ml", "size_group", "dice"]
    ].sort_values("lesion_volume_ml").to_string(index=False)
)

print("\n========================================")
print(" VALIDATION MACRO DICE")
print("========================================")

macro_dice = lesion_df["dice"].mean()

print(f"Lesion-positive Macro Dice : {macro_dice:.4f}")

print("\n--- Lesion Size별 Macro Dice ---")

size_summary = (
    lesion_df
    .groupby("size_group")
    .agg(
        cases=("case", "count"),
        volume_min_ml=("lesion_volume_ml", "min"),
        volume_max_ml=("lesion_volume_ml", "max"),
        macro_dice=("dice", "mean")
    )
    .reindex(["Small", "Medium", "Large"])
)

display(size_summary.round(4))

print("\n--- Empty-GT 확인 ---")
display(
    df_val[df_val["gt_voxels"] == 0][
        ["case", "gt_voxels", "pred_voxels", "dice"]
    ]
)

 nnU-Net VALIDATION CASE-WISE RESULT
전체 Validation Case : 38
병변 존재 Case       : 37
Empty-GT Case        : 1

환자별 Dice
              case  lesion_volume_ml size_group     dice
sub-strokecase0170          0.000000   Empty-GT      NaN
sub-strokecase0138          0.200000      Small 0.780488
sub-strokecase0234          0.232000      Small 0.653061
sub-strokecase0093          0.336000      Small 0.796117
sub-strokecase0088          0.440000      Small 0.873786
sub-strokecase0012          0.448000      Small 0.829268
sub-strokecase0032          0.536000      Small 0.904762
sub-strokecase0059          0.696000      Small 0.896970
sub-strokecase0002          0.944000      Small 0.813793
sub-strokecase0018          1.064000      Small 0.774194
sub-strokecase0098          1.384000      Small 0.419214
sub-strokecase0089          1.536000      Small 0.831409
sub-strokecase0148          2.280000      Small 0.824962
sub-strokecase0142          2.568000      Small 0.902479
sub-strokecase0241         

,cases,volume_min_ml,volume_max_ml,macro_dice
size_group,,,,
Small,16,0.200,4.3859,0.7581
Medium,15,5.224,18.9076,0.8361
Large,6,28.416,107.7089,0.8730



--- Empty-GT 확인 ---


,case,gt_voxels,pred_voxels,dice
25,sub-strokecase0170,0,0,NaN


# [I-N10 Attention Gate Validation 평가]

학습 종료 후 `checkpoint_best.pth`로 Validation 38 Case 전체에 대해  
환자별 full-volume inference를 수행합니다.

평가 항목:
- 전체 Validation 38 Case Case-wise Macro Dice
- 병변 존재 37 Case Macro Dice
- Empty-GT 1 Case 예측 voxel
- Small `<5 mL`
- Medium `5 ≤ volume <20 mL`
- Large `≥20 mL`

> I-N00 baseline과 동일한 split / preprocessing / 평가 기준을 사용합니다.


In [34]:
from google.colab import auth, drive
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import tempfile
import hashlib
import shutil
import json

# 현재 연결된 Drive 경로 사용
drive_roots = [
    Path("/content/gdrive/MyDrive"),
]
drive_root = next((p for p in drive_roots if p.is_dir()), None)

if drive_root is None:
    drive.mount("/content/gdrive")
    drive_root = Path("/content/gdrive/MyDrive")

auth.authenticate_user()

GCS_ROOT = (
    "gs://brainon-cdss-datasets/ischemia/v1/nnunet/"
    "nnUNet_preprocessed/Dataset502_ISLES2022"
)

names = [
    "dataset.json",
    "dataset_fingerprint.json",
    "nnUNetPlans.json",
    "splits_final.json",
]

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
destination = (
    drive_root
    / "brainCDSS/checkpoints/I-N10/metadata_backups"
    / stamp
)

with tempfile.TemporaryDirectory(prefix="nnunet_metadata_") as tmp:
    temporary = Path(tmp)
    contents = {}

    # 네 파일 모두 정상 다운로드·JSON 읽기에 성공해야 Drive에 저장
    for name in names:
        source = f"{GCS_ROOT}/{name}"
        path = temporary / name

        print(f"가져오는 중: {name}", flush=True)
        subprocess.run(
            ["gcloud", "storage", "cp", source, str(path)],
            check=True,
        )
        contents[name] = json.loads(path.read_text(encoding="utf-8"))

    # 노트북에서 확인한 데이터 채널·분할과 대조
    channels = contents["dataset.json"].get("channel_names")
    splits = contents["splits_final.json"]

    if channels != {"0": "DWI", "1": "ADC"}:
        raise ValueError(f"노트북 기록과 입력 채널이 다릅니다: {channels}")

    if not isinstance(splits, list) or len(splits) != 1:
        raise ValueError("노트북 기록의 Fold 1개와 다릅니다.")

    train_ids = splits[0]["train"]
    val_ids = splits[0]["val"]

    if (len(train_ids), len(val_ids)) != (175, 38):
        raise ValueError("노트북의 Train 175 / Validation 38과 다릅니다.")

    if len(set(train_ids)) != len(train_ids):
        raise ValueError("Train 목록에 중복 ID가 있습니다.")
    if len(set(val_ids)) != len(val_ids):
        raise ValueError("Validation 목록에 중복 ID가 있습니다.")
    if set(train_ids) & set(val_ids):
        raise ValueError("Train과 Validation에 겹치는 ID가 있습니다.")

    destination.mkdir(parents=True, exist_ok=False)

    print("\n===== Drive 보존 및 복사 검증 =====")
    for name in names:
        source = temporary / name
        target = destination / name
        shutil.copy2(source, target)

        source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
        target_hash = hashlib.sha256(target.read_bytes()).hexdigest()

        if source_hash != target_hash:
            raise RuntimeError(f"복사 검증 실패: {name}")

        print(f"[정상] {name} — {target.stat().st_size:,} bytes")

print("\n입력 채널:", channels)
print("Fold 수:", len(splits))
print("Train:", len(train_ids), "/ Validation:", len(val_ids))
print("\n저장 위치:", destination)
print("\n학습 실행·기존 파일 덮어쓰기·Git 커밋은 하지 않았습니다.")

가져오는 중: dataset.json
가져오는 중: dataset_fingerprint.json
가져오는 중: nnUNetPlans.json
가져오는 중: splits_final.json

===== Drive 보존 및 복사 검증 =====
[정상] dataset.json — 176 bytes
[정상] dataset_fingerprint.json — 34,159 bytes
[정상] nnUNetPlans.json — 10,567 bytes
[정상] splits_final.json — 6,234 bytes

입력 채널: {'0': 'DWI', '1': 'ADC'}
Fold 수: 1
Train: 175 / Validation: 38

저장 위치: /content/gdrive/MyDrive/brainCDSS/checkpoints/I-N10/metadata_backups/20260903T072553_133942Z

학습 실행·기존 파일 덮어쓰기·Git 커밋은 하지 않았습니다.


In [35]:
# ============================================================
# I-N10 Attention Gate
# Validation 38 Case 종합 성능 평가
#
# 주의:
# - 학습 다시 안 함
# - inference 다시 안 함
# - 이미 생성된 Validation prediction NIfTI + GT를 사용
# - checkpoint_best.pth --val --val_best 결과 기준
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import math

import nibabel as nib
import numpy as np
import pandas as pd

from scipy.ndimage import (
    label as cc_label,
    binary_erosion,
    distance_transform_edt,
)


# ============================================================
# 1. Validation 결과 경로 찾기
# ============================================================

RESULT_ROOT = Path(
    "/content/brainon_cache/nnUNet_results/"
    "Dataset502_ISLES2022/"
    "nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/"
    "fold_0"
)

SUMMARY_PATH = RESULT_ROOT / "validation" / "summary.json"

if not SUMMARY_PATH.is_file():

    candidates = list(
        RESULT_ROOT.glob("**/summary.json")
    )

    if len(candidates) == 1:
        SUMMARY_PATH = candidates[0]

    elif len(candidates) == 0:
        raise FileNotFoundError(
            "I-N10 Validation summary.json을 찾을 수 없습니다.\n"
            "학습은 다시 할 필요 없고, "
            "'--val --val_best' 평가가 완료됐는지 확인하세요."
        )

    else:
        print("summary.json 후보:")
        for p in candidates:
            print(" -", p)

        raise RuntimeError(
            "summary.json 후보가 여러 개입니다."
        )


VAL_DIR = SUMMARY_PATH.parent

print("Validation directory :", VAL_DIR)
print("Summary              :", SUMMARY_PATH)


# ============================================================
# 2. summary.json 읽기
# ============================================================

with SUMMARY_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    summary = json.load(f)


case_items = summary.get(
    "metric_per_case",
    []
)

print("Validation Case 수   :", len(case_items))

if len(case_items) != 38:
    raise RuntimeError(
        f"Validation Case가 38개가 아닙니다: "
        f"{len(case_items)}"
    )


# ============================================================
# 3. 기본 metric 함수
# ============================================================

def safe_divide(
    numerator,
    denominator,
):
    if denominator == 0:
        return np.nan

    return float(
        numerator / denominator
    )


def calculate_voxel_metrics(
    pred,
    gt,
):
    """
    환자 1명 전체 volume 기준 voxel-level metrics.
    """

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    tp = int(
        np.logical_and(
            pred,
            gt,
        ).sum()
    )

    fp = int(
        np.logical_and(
            pred,
            ~gt,
        ).sum()
    )

    fn = int(
        np.logical_and(
            ~pred,
            gt,
        ).sum()
    )

    tn = int(
        np.logical_and(
            ~pred,
            ~gt,
        ).sum()
    )

    pred_positive = tp + fp
    gt_positive = tp + fn

    # --------------------------------------------------------
    # Dice
    #
    # ISLES 방식:
    # GT empty + Pred empty -> 1
    # 한쪽만 empty -> 0
    # --------------------------------------------------------

    dice_den = (
        2 * tp
        + fp
        + fn
    )

    if dice_den == 0:
        dice = 1.0
    else:
        dice = (
            2.0 * tp
            / dice_den
        )

    # --------------------------------------------------------
    # IoU / Jaccard
    # --------------------------------------------------------

    iou_den = (
        tp
        + fp
        + fn
    )

    if iou_den == 0:
        iou = 1.0
    else:
        iou = (
            tp
            / iou_den
        )

    # --------------------------------------------------------
    # Precision / PPV
    # --------------------------------------------------------

    precision = safe_divide(
        tp,
        tp + fp,
    )

    # --------------------------------------------------------
    # Recall / Sensitivity
    # --------------------------------------------------------

    recall = safe_divide(
        tp,
        tp + fn,
    )

    # --------------------------------------------------------
    # Specificity
    # --------------------------------------------------------

    specificity = safe_divide(
        tn,
        tn + fp,
    )

    # --------------------------------------------------------
    # Negative Predictive Value
    # --------------------------------------------------------

    npv = safe_divide(
        tn,
        tn + fn,
    )

    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy = safe_divide(
        tp + tn,
        tp + tn + fp + fn,
    )

    # --------------------------------------------------------
    # False Positive Rate
    # --------------------------------------------------------

    fpr = safe_divide(
        fp,
        fp + tn,
    )

    # --------------------------------------------------------
    # False Negative Rate
    # --------------------------------------------------------

    fnr = safe_divide(
        fn,
        fn + tp,
    )

    # voxel-level binary F1
    # binary segmentation에서는 Dice와 동일
    voxel_f1 = dice

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,

        "gt_voxels":
            gt_positive,

        "pred_voxels":
            pred_positive,

        "dice":
            float(dice),

        "iou":
            float(iou),

        "precision":
            precision,

        "recall_sensitivity":
            recall,

        "specificity":
            specificity,

        "npv":
            npv,

        "accuracy":
            accuracy,

        "fpr":
            fpr,

        "fnr":
            fnr,

        "voxel_f1":
            float(voxel_f1),
    }


# ============================================================
# 4. ISLES 2022 공식 lesion-wise metrics
# ============================================================

def calculate_lesion_metrics(
    pred,
    gt,
):
    """
    ISLES 2022 공식 평가 방식과 같은 개념.

    3D connected components를 사용.

    TP lesion:
      GT component가 prediction과 최소 1 voxel 이상 겹침

    FN lesion:
      GT component가 prediction과 전혀 겹치지 않음

    FP lesion:
      Prediction component가 GT와 전혀 겹치지 않음

    Lesion-wise F1:
      2TP / (2TP + FP + FN)

    둘 다 병변 없음:
      F1 = 1
    """

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    # ISLES 2022 lesion-wise 평가 방식에 맞춰
    # 3x3x3 all-ones connectivity 사용
    structure = np.ones(
        (3, 3, 3),
        dtype=np.uint8,
    )

    labeled_gt, gt_count = (
        cc_label(
            gt,
            structure=structure,
        )
    )

    labeled_pred, pred_count = (
        cc_label(
            pred,
            structure=structure,
        )
    )

    intersection = np.logical_and(
        gt,
        pred,
    )

    lesion_tp = 0
    lesion_fn = 0

    # --------------------------------------------------------
    # GT lesion 기준 TP / FN
    # --------------------------------------------------------

    for lesion_id in range(
        1,
        gt_count + 1,
    ):

        lesion = (
            labeled_gt
            == lesion_id
        )

        detected = np.logical_and(
            lesion,
            intersection,
        ).any()

        if detected:
            lesion_tp += 1
        else:
            lesion_fn += 1

    # --------------------------------------------------------
    # Prediction lesion 기준 FP
    # --------------------------------------------------------

    lesion_fp = 0

    for lesion_id in range(
        1,
        pred_count + 1,
    ):

        lesion = (
            labeled_pred
            == lesion_id
        )

        overlaps_gt = np.logical_and(
            lesion,
            gt,
        ).any()

        if not overlaps_gt:
            lesion_fp += 1

    # --------------------------------------------------------
    # Lesion-wise Precision / Recall / F1
    # --------------------------------------------------------

    if (
        lesion_tp
        + lesion_fp
        + lesion_fn
    ) == 0:

        lesion_f1 = 1.0

    else:

        lesion_f1 = (
            2.0
            * lesion_tp
            / (
                2.0 * lesion_tp
                + lesion_fp
                + lesion_fn
            )
        )

    lesion_precision = safe_divide(
        lesion_tp,
        lesion_tp + lesion_fp,
    )

    lesion_recall = safe_divide(
        lesion_tp,
        lesion_tp + lesion_fn,
    )

    # 둘 다 empty라면
    # detection 관점에서는 완전 정답
    if (
        gt_count == 0
        and pred_count == 0
    ):
        lesion_precision = 1.0
        lesion_recall = 1.0

    # ISLES 공식 metric:
    # Absolute Lesion Count Difference
    abs_lesion_count_difference = abs(
        int(pred_count)
        - int(gt_count)
    )

    return {
        "gt_lesion_count":
            int(gt_count),

        "pred_lesion_count":
            int(pred_count),

        "lesion_tp":
            int(lesion_tp),

        "lesion_fp":
            int(lesion_fp),

        "lesion_fn":
            int(lesion_fn),

        "lesion_precision":
            lesion_precision,

        "lesion_recall":
            lesion_recall,

        "lesion_f1":
            float(lesion_f1),

        "absolute_lesion_count_difference":
            int(
                abs_lesion_count_difference
            ),
    }


# ============================================================
# 5. Surface metrics
#    HD95 / ASSD
# ============================================================

def calculate_surface_metrics(
    pred,
    gt,
    spacing,
):
    """
    HD95 / ASSD 계산.

    단위: mm

    GT와 Pred 둘 다 존재하는 경우에만
    정상적인 surface distance 계산 가능.

    둘 다 empty:
      NaN

    한쪽만 empty:
      inf
    """

    pred = pred.astype(bool)
    gt = gt.astype(bool)

    pred_exists = bool(
        pred.any()
    )

    gt_exists = bool(
        gt.any()
    )

    if (
        not pred_exists
        and not gt_exists
    ):
        return {
            "hd95_mm": np.nan,
            "assd_mm": np.nan,
        }

    if (
        pred_exists
        != gt_exists
    ):
        return {
            "hd95_mm": np.inf,
            "assd_mm": np.inf,
        }

    structure = np.ones(
        (3, 3, 3),
        dtype=bool,
    )

    pred_surface = np.logical_xor(
        pred,
        binary_erosion(
            pred,
            structure=structure,
            border_value=0,
        ),
    )

    gt_surface = np.logical_xor(
        gt,
        binary_erosion(
            gt,
            structure=structure,
            border_value=0,
        ),
    )

    distance_to_gt = (
        distance_transform_edt(
            ~gt_surface,
            sampling=spacing,
        )
    )

    distance_to_pred = (
        distance_transform_edt(
            ~pred_surface,
            sampling=spacing,
        )
    )

    pred_to_gt = (
        distance_to_gt[
            pred_surface
        ]
    )

    gt_to_pred = (
        distance_to_pred[
            gt_surface
        ]
    )

    distances = np.concatenate(
        [
            pred_to_gt,
            gt_to_pred,
        ]
    )

    if distances.size == 0:
        return {
            "hd95_mm": np.nan,
            "assd_mm": np.nan,
        }

    hd95 = float(
        np.percentile(
            distances,
            95,
        )
    )

    assd = float(
        distances.mean()
    )

    return {
        "hd95_mm":
            hd95,

        "assd_mm":
            assd,
    }


# ============================================================
# 6. 환자 38명 각각 계산
# ============================================================

rows = []

print()
print("=" * 80)
print(
    "I-N10 Attention Gate "
    "- Validation 38 Case 평가 시작"
)
print("=" * 80)

for index, item in enumerate(
    case_items,
    start=1,
):

    pred_path = Path(
        item["prediction_file"]
    )

    gt_path = Path(
        item["reference_file"]
    )

    if not pred_path.is_file():
        raise FileNotFoundError(
            f"Prediction 없음:\n"
            f"{pred_path}"
        )

    if not gt_path.is_file():
        raise FileNotFoundError(
            f"Ground Truth 없음:\n"
            f"{gt_path}"
        )

    case_id = (
        gt_path.name
        .replace(".nii.gz", "")
    )

    pred_nii = nib.load(
        pred_path
    )

    gt_nii = nib.load(
        gt_path
    )

    pred = (
        np.asanyarray(
            pred_nii.dataobj
        )
        > 0
    )

    gt = (
        np.asanyarray(
            gt_nii.dataobj
        )
        > 0
    )

    if pred.shape != gt.shape:
        raise RuntimeError(
            f"{case_id}: "
            f"Pred / GT shape 불일치\n"
            f"{pred.shape} != {gt.shape}"
        )

    # --------------------------------------------------------
    # spacing
    # --------------------------------------------------------

    spacing = tuple(
        float(x)
        for x in
        gt_nii.header.get_zooms()[:3]
    )

    # NIfTI affine 기준 voxel volume
    voxel_volume_mm3 = abs(
        np.linalg.det(
            gt_nii.affine[:3, :3]
        )
    )

    voxel_volume_ml = (
        voxel_volume_mm3
        / 1000.0
    )

    # --------------------------------------------------------
    # Voxel metrics
    # --------------------------------------------------------

    voxel_metrics = (
        calculate_voxel_metrics(
            pred,
            gt,
        )
    )

    # --------------------------------------------------------
    # Lesion metrics
    # --------------------------------------------------------

    lesion_metrics = (
        calculate_lesion_metrics(
            pred,
            gt,
        )
    )

    # --------------------------------------------------------
    # Surface metrics
    # --------------------------------------------------------

    surface_metrics = (
        calculate_surface_metrics(
            pred,
            gt,
            spacing,
        )
    )

    # --------------------------------------------------------
    # Volume metrics
    # --------------------------------------------------------

    gt_volume_ml = (
        voxel_metrics[
            "gt_voxels"
        ]
        * voxel_volume_ml
    )

    pred_volume_ml = (
        voxel_metrics[
            "pred_voxels"
        ]
        * voxel_volume_ml
    )

    # ISLES 공식 metric:
    # Absolute Volume Difference
    absolute_volume_difference_ml = abs(
        pred_volume_ml
        - gt_volume_ml
    )

    signed_volume_error_ml = (
        pred_volume_ml
        - gt_volume_ml
    )

    if gt_volume_ml > 0:

        relative_volume_error_pct = (
            signed_volume_error_ml
            / gt_volume_ml
            * 100.0
        )

        absolute_relative_volume_error_pct = (
            absolute_volume_difference_ml
            / gt_volume_ml
            * 100.0
        )

    else:

        relative_volume_error_pct = (
            np.nan
        )

        absolute_relative_volume_error_pct = (
            np.nan
        )

    # --------------------------------------------------------
    # 병변 크기 그룹
    # --------------------------------------------------------

    if gt_volume_ml == 0:

        size_group = "Empty-GT"

    elif gt_volume_ml < 5:

        size_group = "Small"

    elif gt_volume_ml < 20:

        size_group = "Medium"

    else:

        size_group = "Large"

    # --------------------------------------------------------
    # 결과 저장
    # --------------------------------------------------------

    row = {
        "case":
            case_id,

        "size_group":
            size_group,

        "spacing_x_mm":
            spacing[0],

        "spacing_y_mm":
            spacing[1],

        "spacing_z_mm":
            spacing[2],

        "voxel_volume_ml":
            voxel_volume_ml,

        **voxel_metrics,

        **lesion_metrics,

        **surface_metrics,

        "gt_volume_ml":
            gt_volume_ml,

        "pred_volume_ml":
            pred_volume_ml,

        "signed_volume_error_ml":
            signed_volume_error_ml,

        "absolute_volume_difference_ml":
            absolute_volume_difference_ml,

        "relative_volume_error_pct":
            relative_volume_error_pct,

        "absolute_relative_volume_error_pct":
            absolute_relative_volume_error_pct,
    }

    rows.append(
        row
    )

    print(
        f"[{index:02d}/38] "
        f"{case_id} | "
        f"Dice={row['dice']:.4f} | "
        f"Lesion-F1={row['lesion_f1']:.4f} | "
        f"GT={row['gt_volume_ml']:.3f}mL | "
        f"Pred={row['pred_volume_ml']:.3f}mL"
    )


# ============================================================
# 7. DataFrame
# ============================================================

df = pd.DataFrame(
    rows
)

df = (
    df
    .sort_values("case")
    .reset_index(drop=True)
)

positive_df = (
    df[
        df["gt_voxels"] > 0
    ]
    .copy()
)

empty_df = (
    df[
        df["gt_voxels"] == 0
    ]
    .copy()
)


if len(df) != 38:
    raise RuntimeError(
        f"최종 Case 수 오류: "
        f"{len(df)}"
    )


# ============================================================
# 8. Mean 계산 helper
# ============================================================

def finite_mean(series):

    values = np.asarray(
        series,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        values.mean()
    )


def finite_median(series):

    values = np.asarray(
        series,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        np.median(values)
    )


# ============================================================
# 9. 전체 요약
# ============================================================

def make_summary(
    data,
    scope_name,
):

    return {
        "scope":
            scope_name,

        "cases":
            len(data),

        # ----------------------------------------------------
        # Segmentation overlap
        # ----------------------------------------------------

        "macro_dice":
            finite_mean(
                data["dice"]
            ),

        "median_dice":
            finite_median(
                data["dice"]
            ),

        "macro_iou":
            finite_mean(
                data["iou"]
            ),

        "macro_precision":
            finite_mean(
                data["precision"]
            ),

        "macro_recall_sensitivity":
            finite_mean(
                data[
                    "recall_sensitivity"
                ]
            ),

        "macro_specificity":
            finite_mean(
                data["specificity"]
            ),

        "macro_npv":
            finite_mean(
                data["npv"]
            ),

        "macro_accuracy":
            finite_mean(
                data["accuracy"]
            ),

        "macro_fpr":
            finite_mean(
                data["fpr"]
            ),

        "macro_fnr":
            finite_mean(
                data["fnr"]
            ),

        # ----------------------------------------------------
        # Surface
        # ----------------------------------------------------

        "mean_hd95_mm":
            finite_mean(
                data["hd95_mm"]
            ),

        "median_hd95_mm":
            finite_median(
                data["hd95_mm"]
            ),

        "mean_assd_mm":
            finite_mean(
                data["assd_mm"]
            ),

        # ----------------------------------------------------
        # ISLES lesion metrics
        # ----------------------------------------------------

        "macro_lesion_f1":
            finite_mean(
                data["lesion_f1"]
            ),

        "macro_lesion_precision":
            finite_mean(
                data[
                    "lesion_precision"
                ]
            ),

        "macro_lesion_recall":
            finite_mean(
                data[
                    "lesion_recall"
                ]
            ),

        "mean_absolute_lesion_count_difference":
            finite_mean(
                data[
                    "absolute_lesion_count_difference"
                ]
            ),

        # ----------------------------------------------------
        # Volume
        # ----------------------------------------------------

        "mean_gt_volume_ml":
            finite_mean(
                data[
                    "gt_volume_ml"
                ]
            ),

        "mean_pred_volume_ml":
            finite_mean(
                data[
                    "pred_volume_ml"
                ]
            ),

        "mean_absolute_volume_difference_ml":
            finite_mean(
                data[
                    "absolute_volume_difference_ml"
                ]
            ),

        "mean_relative_volume_error_pct":
            finite_mean(
                data[
                    "relative_volume_error_pct"
                ]
            ),

        "mean_absolute_relative_volume_error_pct":
            finite_mean(
                data[
                    "absolute_relative_volume_error_pct"
                ]
            ),
    }


summary_df = pd.DataFrame(
    [
        make_summary(
            df,
            "All Validation 38",
        ),

        make_summary(
            positive_df,
            "Lesion-positive 37",
        ),
    ]
)


# ============================================================
# 10. 병변 크기별 요약
# ============================================================

size_order = [
    "Small",
    "Medium",
    "Large",
]

size_rows = []

for group in size_order:

    group_df = positive_df[
        positive_df[
            "size_group"
        ]
        == group
    ]

    if len(group_df) == 0:
        continue

    result = make_summary(
        group_df,
        group,
    )

    size_rows.append(
        result
    )


size_summary_df = pd.DataFrame(
    size_rows
)


# ============================================================
# 11. 핵심 결과 출력
# ============================================================

print()
print()
print("=" * 90)
print(
    "I-N10 Attention Gate "
    "- Validation 최종 성능"
)
print("=" * 90)

print(
    f"Validation 전체       : "
    f"{len(df)} Case"
)

print(
    f"병변 존재             : "
    f"{len(positive_df)} Case"
)

print(
    f"Empty-GT              : "
    f"{len(empty_df)} Case"
)

print()

print(
    "All-38 Macro Dice     : "
    f"{df['dice'].mean():.6f}"
)

print(
    "Positive-37 Macro Dice: "
    f"{positive_df['dice'].mean():.6f}"
)

print(
    "All-38 Lesion-wise F1 : "
    f"{df['lesion_f1'].mean():.6f}"
)

print(
    "Positive Lesion F1    : "
    f"{positive_df['lesion_f1'].mean():.6f}"
)

print(
    "Mean AVD (mL)         : "
    f"{df['absolute_volume_difference_ml'].mean():.6f}"
)

print(
    "Mean ALD              : "
    f"{df['absolute_lesion_count_difference'].mean():.6f}"
)

print(
    "Mean HD95 (finite)    : "
    f"{finite_mean(df['hd95_mm']):.6f} mm"
)

print(
    "Mean ASSD (finite)    : "
    f"{finite_mean(df['assd_mm']):.6f} mm"
)

print("=" * 90)


# ============================================================
# 12. 전체 요약표
# ============================================================

print()
print(
    "[1] 전체 / 병변 존재 Case 요약"
)

display(
    summary_df.round(6)
)


# ============================================================
# 13. 환자별 전체 성능표
# ============================================================

patient_columns = [

    "case",
    "size_group",

    # Volume
    "gt_volume_ml",
    "pred_volume_ml",
    "absolute_volume_difference_ml",
    "relative_volume_error_pct",

    # Segmentation
    "dice",
    "iou",
    "precision",
    "recall_sensitivity",
    "specificity",
    "npv",
    "accuracy",
    "fpr",
    "fnr",

    # Boundary
    "hd95_mm",
    "assd_mm",

    # Lesion detection
    "gt_lesion_count",
    "pred_lesion_count",
    "lesion_tp",
    "lesion_fp",
    "lesion_fn",
    "lesion_precision",
    "lesion_recall",
    "lesion_f1",
    "absolute_lesion_count_difference",
]

print()
print(
    "[2] 환자별 전체 성능지표"
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.width",
    300,
):

    display(
        df[
            patient_columns
        ].round(6)
    )


# ============================================================
# 14. Small / Medium / Large
# ============================================================

print()
print(
    "[3] 병변 크기별 성능"
)

display(
    size_summary_df.round(6)
)


# ============================================================
# 15. Empty-GT
# ============================================================

print()
print(
    "[4] Empty-GT Case"
)

display(
    empty_df[
        [
            "case",
            "gt_voxels",
            "pred_voxels",
            "gt_lesion_count",
            "pred_lesion_count",
            "gt_volume_ml",
            "pred_volume_ml",
            "dice",
            "lesion_f1",
            "absolute_volume_difference_ml",
            "absolute_lesion_count_difference",
        ]
    ].round(6)
)


# ============================================================
# 16. 낮은 Dice 순서 확인
# ============================================================

print()
print(
    "[5] Dice 낮은 Case 순서"
)

display(
    df[
        [
            "case",
            "size_group",
            "dice",
            "lesion_f1",
            "recall_sensitivity",
            "precision",
            "hd95_mm",
            "gt_volume_ml",
            "pred_volume_ml",
        ]
    ]
    .sort_values(
        "dice",
        ascending=True,
    )
    .round(6)
)


# ============================================================
# 17. CSV 저장
# ============================================================

drive_roots = [
    Path(
        "/content/gdrive/MyDrive"
    ),
    Path(
        "/content/drive/MyDrive"
    ),
]

drive_root = next(
    (
        p
        for p in drive_roots
        if p.is_dir()
    ),
    None,
)

if drive_root is not None:

    timestamp = datetime.now(
        timezone.utc
    ).strftime(
        "%Y%m%dT%H%M%S_%fZ"
    )

    save_dir = (
        drive_root
        / "brainCDSS"
        / "results"
        / "I-N10"
        / "validation"
        / f"metrics_{timestamp}"
    )

    save_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    df.to_csv(
        save_dir
        / "validation_case_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary_df.to_csv(
        save_dir
        / "validation_metrics_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    size_summary_df.to_csv(
        save_dir
        / "validation_size_group_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )

    print()
    print(
        "CSV 저장 완료:"
    )
    print(
        save_dir
    )

else:

    print()
    print(
        "Google Drive mount를 찾지 못해서 "
        "화면 출력만 완료했습니다."
    )


print()
print("=" * 90)
print("평가 완료")
print("=" * 90)

Validation directory : /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0/validation
Summary              : /content/brainon_cache/nnUNet_results/Dataset502_ISLES2022/nnUNetTrainerIschemiaAttention__nnUNetPlans__3d_fullres/fold_0/validation/summary.json
Validation Case 수   : 38

I-N10 Attention Gate - Validation 38 Case 평가 시작
[01/38] sub-strokecase0002 | Dice=0.8138 | Lesion-F1=1.0000 | GT=0.944mL | Pred=1.376mL
[02/38] sub-strokecase0012 | Dice=0.8293 | Lesion-F1=1.0000 | GT=0.448mL | Pred=0.536mL
[03/38] sub-strokecase0018 | Dice=0.7742 | Lesion-F1=1.0000 | GT=1.064mL | Pred=0.672mL
[04/38] sub-strokecase0021 | Dice=0.8274 | Lesion-F1=0.8182 | GT=17.720mL | Pred=16.296mL
[05/38] sub-strokecase0025 | Dice=0.9025 | Lesion-F1=1.0000 | GT=7.752mL | Pred=8.576mL
[06/38] sub-strokecase0032 | Dice=0.9048 | Lesion-F1=1.0000 | GT=0.536mL | Pred=0.472mL
[07/38] sub-strokecase0040 | Dice=0.8035 | Lesion-F1=0.7619 | GT=10.616

,scope,cases,macro_dice,median_dice,macro_iou,macro_precision,macro_recall_sensitivity,macro_specificity,macro_npv,macro_accuracy,...,mean_assd_mm,macro_lesion_f1,macro_lesion_precision,macro_lesion_recall,mean_absolute_lesion_count_difference,mean_gt_volume_ml,mean_pred_volume_ml,mean_absolute_volume_difference_ml,mean_relative_volume_error_pct,mean_absolute_relative_volume_error_pct
0,All Validation 38,38,0.813412,0.848067,0.705807,0.887225,0.784638,0.999814,0.999619,0.999437,...,1.934935,0.800391,0.923063,0.761525,2.763158,15.775152,14.347746,2.320084,-9.356365,23.919542
1,Lesion-positive 37,37,0.808369,0.832719,0.697856,0.887225,0.784638,0.999809,0.999609,0.999421,...,1.934935,0.794996,0.920983,0.755080,2.837838,16.201508,14.735523,2.382789,-9.356365,23.919542



[2] 환자별 전체 성능지표


,case,size_group,gt_volume_ml,pred_volume_ml,absolute_volume_difference_ml,relative_volume_error_pct,dice,iou,precision,recall_sensitivity,specificity,npv,accuracy,fpr,fnr,hd95_mm,assd_mm,gt_lesion_count,pred_lesion_count,lesion_tp,lesion_fp,lesion_fn,lesion_precision,lesion_recall,lesion_f1,absolute_lesion_count_difference
0,sub-strokecase0002,Small,0.944000,1.376000,0.432000,45.762712,0.813793,0.686047,0.686047,1.000000,0.999940,1.000000,0.999940,0.000060,0.000000,2.000000,0.452946,1,1,1,0,0,1.000000,1.000000,1.000000,0
1,sub-strokecase0012,Small,0.448000,0.536000,0.088000,19.642857,0.829268,0.708333,0.761194,0.910714,0.999982,0.999994,0.999977,0.000018,0.089286,2.000000,0.341463,1,1,1,0,0,1.000000,1.000000,1.000000,0
2,sub-strokecase0018,Small,1.064000,0.672000,0.392000,-36.842105,0.774194,0.631579,1.000000,0.631579,1.000000,0.999946,0.999946,0.000000,0.368421,2.828427,0.631805,1,1,1,0,0,1.000000,1.000000,1.000000,0
3,sub-strokecase0021,Medium,17.720001,16.296001,1.424000,-8.036117,0.827375,0.705576,0.863525,0.794131,0.999691,0.999494,0.999187,0.000309,0.205869,18.127285,1.955645,10,13,9,3,1,0.750000,0.900000,0.818182,3
4,sub-strokecase0025,Medium,7.752000,8.576000,0.824000,10.629515,0.902499,0.822321,0.859142,0.950464,0.999835,0.999948,0.999783,0.000165,0.049536,2.000000,0.337450,4,4,4,0,0,1.000000,1.000000,1.000000,0
5,sub-strokecase0032,Small,0.536000,0.472000,0.064000,-11.940299,0.904762,0.826087,0.966102,0.850746,0.999998,0.999989,0.999987,0.000002,0.149254,2.000000,0.208000,1,1,1,0,0,1.000000,1.000000,1.000000,0
6,sub-strokecase0040,Medium,10.616000,7.504000,3.112000,-29.314243,0.803532,0.671587,0.970149,0.685757,0.999969,0.999538,0.999507,0.000031,0.314243,4.472136,0.824300,12,11,8,1,4,0.888889,0.666667,0.761905,1
7,sub-strokecase0041,Medium,5.224000,4.312000,0.912000,-17.457887,0.877517,0.781764,0.970315,0.800919,0.999983,0.999858,0.999841,0.000017,0.199081,2.000000,1.243396,10,6,6,0,4,1.000000,0.600000,0.750000,4
8,sub-strokecase0050,Medium,15.399999,7.831999,7.567999,-49.142857,0.670799,0.504663,0.994893,0.505974,0.999994,0.998946,0.998942,0.000006,0.494026,7.112088,1.665468,11,14,6,1,5,0.857143,0.545455,0.666667,3
9,sub-strokecase0059,Small,0.696000,0.624000,0.072000,-10.344828,0.896970,0.813187,0.948718,0.850575,0.999996,0.999986,0.999981,0.000004,0.149425,2.000000,0.233129,1,1,1,0,0,1.000000,1.000000,1.000000,0



[3] 병변 크기별 성능


...


[4] Empty-GT Case


,case,gt_voxels,pred_voxels,gt_lesion_count,pred_lesion_count,gt_volume_ml,pred_volume_ml,dice,lesion_f1,absolute_volume_difference_ml,absolute_lesion_count_difference
25,sub-strokecase0170,0,0,0,0,0.0,0.0,1.0,1.0,0.0,0



[5] Dice 낮은 Case 순서


,case,size_group,dice,lesion_f1,recall_sensitivity,precision,hd95_mm,gt_volume_ml,pred_volume_ml
10,sub-strokecase0068,Small,0.166337,0.500000,0.091106,0.954545,46.805897,3.688000,0.352000
18,sub-strokecase0098,Small,0.419214,0.545455,0.277457,0.857143,39.989931,1.384000,0.448000
11,sub-strokecase0076,Medium,0.620874,0.590909,0.459709,0.956044,23.194707,6.056000,2.912000
36,sub-strokecase0234,Small,0.653061,0.800000,0.551724,0.800000,32.203444,0.232000,0.160000
8,sub-strokecase0050,Medium,0.670799,0.666667,0.505974,0.994893,7.112088,15.399999,7.831999
2,sub-strokecase0018,Small,0.774194,1.000000,0.631579,1.000000,2.828427,1.064000,0.672000
17,sub-strokecase0097,Medium,0.774908,0.823529,0.807692,0.744681,4.000000,7.280000,7.896000
15,sub-strokecase0092,Large,0.775628,0.750000,0.701720,0.866936,6.000000,66.032005,53.448004
20,sub-strokecase0138,Small,0.780488,1.000000,0.640000,1.000000,2.000000,0.200000,0.128000
16,sub-strokecase0093,Small,0.796117,1.000000,0.976190,0.672131,2.000000,0.336000,0.488000



CSV 저장 완료:
/content/gdrive/MyDrive/brainCDSS/results/I-N10/validation/metrics_20260903T080305_797965Z

평가 완료
